# Session 4 - Memory & Prompt Engineering

AI Agents & Workflows: Basics

In this notebook we will explore:

- why LLMs are stateless
- how conversation history works
- how LangChain represents messages
- how to build prompt templates
- how to add conversation history to prompts
- how short-term memory works with LangChain agents
- practical prompt engineering patterns

## 0. Setup

This notebook expects:

- `.env` file in the project root
- `OPENAI_API_KEY` inside `.env`
- optional `OPENAI_MODEL` inside `.env`

Example:

```bash
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-4o-mini
```

If you are behind a corporate proxy, keep proxy settings only in your local `.env`, not in Git.

In [ ]:
import os
from dotenv import load_dotenv

# The notebook is inside notebooks/, so we load ../.env from the project root.
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print("OPENAI_API_KEY loaded:", bool(api_key))
print("Model:", model_name)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=model_name,
    temperature=0
)

print("LLM initialized")

## 1. LLMs Are Stateless

A direct model call does not automatically remember previous calls.

This is one of the most important concepts in AI application development.

The model receives input, generates output, and then the call is finished.

Memory must be managed by the application.

In [ ]:
# First independent call
response_1 = llm.invoke("My name is Vlad. Reply with one short sentence.")
print(response_1.content)

In [ ]:
# Second independent call
# The model does not automatically receive the previous call.
response_2 = llm.invoke("What is my name? Reply with one short sentence.")
print(response_2.content)

### Discussion

The second call may not know the name because we did not send the previous message.

This means:

```text
Memory does not live inside the model call.
Memory is context that the application sends to the model.
```

## 2. LangChain Messages

LangChain works with structured messages, not only plain strings.

Common message types:

- `SystemMessage`
- `HumanMessage`
- `AIMessage`
- `ToolMessage`

This makes conversations easier to manage.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="My name is Vlad."),
    AIMessage(content="Nice to meet you, Vlad."),
    HumanMessage(content="What is my name?")
]

for message in messages:
    print(type(message).__name__, "=>", message.content)

In [ ]:
# Now we explicitly send the whole message history to the model.
response = llm.invoke(messages)
print(response.content)

## 3. Manual Conversation Memory

The simplest form of memory is a Python list.

We store messages in a list and send the full list on every model call.

This is not the best production solution, but it is very useful for understanding the concept.

In [ ]:
conversation_history = [
    SystemMessage(content="You are a helpful assistant. Keep answers short.")
]

def ask_with_manual_memory(question: str) -> str:
    conversation_history.append(HumanMessage(content=question))

    response = llm.invoke(conversation_history)

    conversation_history.append(AIMessage(content=response.content))

    return response.content

In [ ]:
print(ask_with_manual_memory("My favorite programming language is Python."))

In [ ]:
print(ask_with_manual_memory("What is my favorite programming language?"))

In [ ]:
for i, message in enumerate(conversation_history, start=1):
    print(f"{i}. {type(message).__name__}: {message.content}")

## 4. Context Window

Every LLM has a maximum amount of input it can process.

The context may include:

- system prompt
- previous user messages
- previous AI messages
- tool results
- retrieved documents
- current user question

More context is not always better.

## 5. Simple Context Management

If the conversation becomes too long, we need to manage context.

A simple approach is to keep only the last N messages.

This is called trimming conversation history.

In [ ]:
def keep_last_messages(messages, max_messages: int = 6):
    """Keep the system message and the last N non-system messages."""
    system_messages = [m for m in messages if isinstance(m, SystemMessage)]
    other_messages = [m for m in messages if not isinstance(m, SystemMessage)]
    return system_messages + other_messages[-max_messages:]


trimmed_history = keep_last_messages(conversation_history, max_messages=4)

for i, message in enumerate(trimmed_history, start=1):
    print(f"{i}. {type(message).__name__}: {message.content}")

## 6. Summary Memory

Another approach is to summarize older messages.

Instead of sending the full history, the application can send:

```text
Summary of old conversation
+
Recent messages
+
Current question
```

This reduces tokens while keeping important context.

In [ ]:
def summarize_messages(messages) -> str:
    text = "\n".join(
        f"{type(m).__name__}: {m.content}"
        for m in messages
        if not isinstance(m, SystemMessage)
    )

    summary_prompt = f"""
Summarize the following conversation in 3 bullet points.
Keep only important user facts and decisions.

Conversation:
{text}
"""

    response = llm.invoke(summary_prompt)
    return response.content


summary = summarize_messages(conversation_history)
print(summary)

## 7. Memory vs RAG

Memory and RAG are related, but not the same.

```text
Memory
  = what happened in the conversation

RAG
  = what external knowledge is relevant
```

Examples:

- Memory: "The user said their favorite language is Python."
- RAG: "Search internal documentation for the Kubernetes onboarding process."

## 8. Prompt Engineering

Prompt engineering means designing instructions and context so the model produces better output.

A strong prompt usually contains:

```text
Role
Task
Context
Constraints
Output Format
```

## 9. Bad Prompt vs Better Prompt

A vague prompt gives the model too much freedom.

A specific prompt gives the model direction.

In [ ]:
bad_prompt = "Explain Docker."

response = llm.invoke(bad_prompt)
print(response.content)

In [ ]:
better_prompt = """
You are a Senior DevOps Engineer.

Explain Docker to a Junior Developer.

Constraints:
- Use simple language
- Use maximum 5 bullet points
- Include one practical example
"""

response = llm.invoke(better_prompt)
print(response.content)

## 10. LangChain Prompt Templates

Instead of hardcoding strings everywhere, LangChain lets us create reusable prompt templates.

This makes prompts easier to maintain, test, and reuse.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} to {audience}. Use maximum {max_bullets} bullet points."
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "topic": "Kubernetes namespaces",
    "audience": "junior developers",
    "max_bullets": 5
})

print(result)

## 11. ChatPromptTemplate with Message Roles

For chat models, it is better to build prompts from message roles.

This separates:

- system instructions
- user request
- conversation history

In [ ]:
role_based_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful technical trainer. Be concise and practical."),
    ("human", "Explain {topic} for {audience}.")
])

chain = role_based_prompt | llm | StrOutputParser()

result = chain.invoke({
    "topic": "LangChain messages",
    "audience": "software engineers"
})

print(result)

## 12. MessagesPlaceholder

`MessagesPlaceholder` allows us to insert conversation history into a prompt.

This is one of the most useful LangChain patterns for conversational applications.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

conversation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation history when relevant."),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

conversation_chain = conversation_prompt | llm | StrOutputParser()

In [ ]:
demo_history = [
    HumanMessage(content="My team works mostly with Jenkins and Kubernetes."),
    AIMessage(content="Got it. I will keep Jenkins and Kubernetes in mind.")
]

result = conversation_chain.invoke({
    "history": demo_history,
    "question": "What technologies does my team use?"
})

print(result)

## 13. Reusable Conversation Function with Prompt Template

Now let's combine:

- `ChatPromptTemplate`
- `MessagesPlaceholder`
- manual history
- model
- output parser

In [ ]:
app_history = []

def ask_with_prompt_template(question: str) -> str:
    result = conversation_chain.invoke({
        "history": app_history,
        "question": question
    })

    app_history.append(HumanMessage(content=question))
    app_history.append(AIMessage(content=result))

    return result

In [ ]:
print(ask_with_prompt_template("My current project is about AI Agents and Workflows."))

In [ ]:
print(ask_with_prompt_template("What is my current project about?"))

## 14. Few-Shot Prompting

Few-shot prompting means giving examples.

This is useful when you want a specific style, classification, or output pattern.

In [ ]:
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify engineering tickets into one category: Build, Deployment, Test, Documentation, Other."),
    ("human", "Ticket: Jenkins pipeline fails during unit tests."),
    ("ai", "Category: Test"),
    ("human", "Ticket: Update Kubernetes onboarding guide."),
    ("ai", "Category: Documentation"),
    ("human", "Ticket: {ticket}")
])

classifier_chain = few_shot_prompt | llm | StrOutputParser()

tickets = [
    "Docker image cannot be pushed to registry.",
    "Add missing README section for local setup.",
    "Production deployment is waiting for approval."
]

for ticket in tickets:
    print(ticket)
    print(classifier_chain.invoke({"ticket": ticket}))
    print("---")

## 15. Structured Output with Prompting

For applications, free text is often hard to parse.

We can ask the model to return structured JSON.

Later, in production, we can validate this with schemas.

In [ ]:
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "You extract structured information. Return only valid JSON."),
    ("human", """
Analyze this incident:

{incident}

Return JSON with these fields:
- summary
- severity
- probable_cause
- next_steps
""")
])

structured_chain = structured_prompt | llm | StrOutputParser()

incident = "The nightly build failed after a dependency update. Unit tests for the authentication module are failing."

result = structured_chain.invoke({"incident": incident})
print(result)

## 16. Tool Messages Concept

When an agent calls a tool, the tool result is also represented as a message.

The flow is:

```text
User asks question
    ↓
LLM requests tool call
    ↓
Application executes tool
    ↓
ToolMessage contains result
    ↓
LLM uses result for final answer
```

In [ ]:
tool_message = ToolMessage(
    content="Build status: FAILED. Failed stage: Unit Tests.",
    tool_call_id="demo-tool-call-1"
)

print(type(tool_message).__name__)
print(tool_message.content)

## 17. LangChain Agent with Short-Term Memory

Modern LangChain agents can use a checkpointer for short-term memory.

The important idea:

```text
thread_id = conversation identity
checkpointer = stores agent state
messages = conversation history
```

This prepares us for LangGraph.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

memory_agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
    system_prompt="You are a helpful assistant. Keep answers short."
)

thread_config = {"configurable": {"thread_id": "session-4-demo"}}

print("Agent with checkpointer created")

In [ ]:
result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Vlad and I am preparing an AI Agents course."}]},
    thread_config
)

print(result["messages"][-1].content)

In [ ]:
result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What course am I preparing?"}]},
    thread_config
)

print(result["messages"][-1].content)

## 18. Thread Isolation

Short-term memory is usually scoped by thread or session.

Different `thread_id` means different conversation memory.

In [ ]:
new_thread_config = {"configurable": {"thread_id": "different-thread"}}

result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What course am I preparing?"}]},
    new_thread_config
)

print(result["messages"][-1].content)

## 19. Practical Prompt Best Practices

Use these rules when building real AI applications:

- keep system prompts small and clear
- define the expected output format
- separate instructions from user data
- avoid conflicting instructions
- do not overload the context
- use examples for difficult formats
- validate important outputs
- version prompts like code

## 20. Enterprise AI Application Structure

A maintainable AI application usually separates concerns:

```text
project/

├── agents/
├── tools/
├── prompts/
├── memory/
├── rag/
├── config/
├── tests/
└── main.py
```

Prompts should not be random strings hidden across the codebase.

## 21. Exercise 1 - Build Your Own Memory Chat

Task:

Create a new conversation function that:

1. stores history in a list
2. sends history to the model
3. keeps only the last 6 messages
4. returns a short answer

In [ ]:
# Exercise template

exercise_history = [
    SystemMessage(content="You are a concise engineering assistant.")
]

def exercise_chat(question: str) -> str:
    # TODO:
    # 1. append HumanMessage
    # 2. trim history
    # 3. call llm
    # 4. append AIMessage
    # 5. return response content

    raise NotImplementedError("Implement this function")

## 22. Exercise 2 - Improve a Prompt

Bad prompt:

```text
Summarize this.
```

Improve it by adding:

- role
- task
- context
- constraints
- output format

In [ ]:
# Exercise prompt template

improved_summary_prompt = """
TODO: Write a better prompt here.
"""

print(improved_summary_prompt)

## 23. Discussion Questions

1. Where does memory live: in the model or in the application?
2. Why is sending the full conversation history not always a good idea?
3. What is the difference between memory and RAG?
4. Why are message roles useful?
5. Why should prompts be versioned and reviewed?
6. What can go wrong if the system prompt is unclear?

## 24. Key Takeaways

- LLM calls are stateless by default.
- Memory is context managed by the application.
- LangChain represents conversations as messages.
- `ChatPromptTemplate` makes prompts reusable.
- `MessagesPlaceholder` inserts conversation history.
- Context window management matters.
- Prompt engineering is software engineering.
- LangGraph is the next step for stateful agent workflows.

## 25. Useful Documentation

Official LangChain documentation:

- Messages: https://docs.langchain.com/oss/python/langchain/messages
- Short-term memory: https://docs.langchain.com/oss/python/langchain/short-term-memory
- Long-term memory: https://docs.langchain.com/oss/python/langchain/long-term-memory
- Agents: https://docs.langchain.com/oss/python/langchain/agents
- Context engineering: https://docs.langchain.com/oss/python/langchain/context-engineering